# PaDIS manual reconstruction on Colab

This notebook prepares a Colab/Colab Pro+ runtime for the PaDIS inference matrix. It authenticates to Google Cloud, mounts `gs://padis-bucket` as `/mnt/data`, clones the current LION repo onto Colab's local disk, installs a local conda environment following the LION setup instructions, and runs the resumable manual reconstruction runner.

The repo checkout and conda environment live under `/content`, so they are fast but disappear when the runtime is destroyed. All outputs, logs, job manifests, and `.done`/`.failed` markers are written under the bucket-backed `/mnt/data` tree. If Colab disconnects, reconnect and rerun the setup/final reconstruction cells; completed jobs will be skipped.

In [ ]:
# Edit these before running if needed.
from pathlib import Path
import shlex

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
    WANDB_API_KEY = userdata.get("WANDB_API_KEY") or ""
except Exception:
    GITHUB_TOKEN = ""
    WANDB_API_KEY = ""

CONFIG = {
    "PADIS_GCS_BUCKET": "padis-bucket",
    "PADIS_DATA_MOUNT": "/mnt/data",
    "LION_ROOT": "/content/LION",
    "LION_GIT_URL": "https://github.com/THartigan/LION.git",
    "LION_GIT_REF": "feature/PaDIS_Implementation",  # branch/tag/commit to fetch and fast-forward
    "GITHUB_TOKEN": GITHUB_TOKEN,  # optional Colab secret for private forks/branches
    "WANDB_API_KEY": WANDB_API_KEY,  # optional Colab secret for WandB artifact logging
    "LION_CONDA_ENV": "lion",
    "MINIFORGE_ROOT": "/content/miniforge3",
    "PADIS_GCP_RUN_NAME": "PaDIS-Reproduction-GCP",
    "PADIS_RECON_METHODS": "all",
    "PADIS_RECON_EXPERIMENTS": "paper_matrix",
    "PADIS_RECON_ABLATIONS": "all",
    "PADIS_RECON_MAX_SAMPLES": "25",
    "PADIS_RECON_TASKS_PER_GPU": "2",
    "PADIS_RECON_SAVE_PREVIEWS": "1",
    "PADIS_RECON_PROG_BAR": "1",
    "PADIS_RAM_DISK": "/mnt/ram-disk",
    "PADIS_RAM_DISK_SIZE": "70%",  # tmpfs cap for staged training data; adjust if Colab RAM is smaller/larger
    "PADIS_MANUAL_RECON_USE_RAMDISK_DATA": "1",
    "PADIS_MANUAL_RECON_CREATE_RAMDISK": "1",
    "PADIS_MANUAL_RECON_REMOVE_RAMDISK_AFTER_TRAINING": "1",
    "PADIS_WANDB_PROJECT": "PaDIS-Reproduction",
    "PADIS_WANDB_MODE": "online",
    "PADIS_NO_WANDB": "0",
    "PADIS_NO_WANDB_ARTIFACT": "0",
    "MPLBACKEND": "Agg",
}

CONFIG["LION_DATA_PATH"] = f"{CONFIG['PADIS_DATA_MOUNT']}/Datasets"
CONFIG["LION_EXPERIMENTS_PATH"] = f"{CONFIG['LION_DATA_PATH']}/experiments"
CONFIG["PADIS_RUN_ROOT"] = f"{CONFIG['LION_EXPERIMENTS_PATH']}/PaDIS"

env_path = Path("/content/padis_colab_env.sh")
env_path.write_text("\n".join(f"export {key}={shlex.quote(value)}" for key, value in CONFIG.items()) + "\n")
for line in env_path.read_text().splitlines():
    if line.startswith("export GITHUB_TOKEN=") and CONFIG["GITHUB_TOKEN"]:
        print("export GITHUB_TOKEN=<redacted>")
    elif line.startswith("export WANDB_API_KEY=") and CONFIG["WANDB_API_KEY"]:
        print("export WANDB_API_KEY=<redacted>")
    else:
        print(line)

## Authenticate to Google Cloud

The first cell uses Colab's browser-based auth. The second shell command checks Application Default Credentials, which `gcsfuse` may need. If it prints an auth URL, open it and paste the code back into the prompt.

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab Google authentication complete.")
except Exception as exc:
    print(f"Colab auth was not available or did not complete: {exc}")

In [ ]:
%%bash
set -euo pipefail
if ! gcloud auth application-default print-access-token >/dev/null 2>&1; then
  echo "Application Default Credentials are not available. Starting browser/no-browser login."
  gcloud auth application-default login --no-browser
fi
gcloud auth list
gcloud auth application-default print-access-token >/dev/null
echo "Application Default Credentials are available."

## Install `gcsfuse` and mount the bucket

This mounts `gs://padis-bucket` as `/mnt/data`, matching the path used by the GCP runners.

In [ ]:
%%bash
set -euo pipefail
source /content/padis_colab_env.sh

if ! command -v gcsfuse >/dev/null 2>&1; then
  sudo apt-get update
  sudo apt-get install -y curl gnupg lsb-release ca-certificates
  curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg \
    | sudo gpg --dearmor -o /usr/share/keyrings/cloud.google.gpg
  export GCSFUSE_REPO="gcsfuse-$(lsb_release -c -s)"
  echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt ${GCSFUSE_REPO} main" \
    | sudo tee /etc/apt/sources.list.d/gcsfuse.list >/dev/null
  sudo apt-get update
  sudo apt-get install -y gcsfuse
fi
if ! command -v zstd >/dev/null 2>&1; then
  sudo apt-get update
  sudo apt-get install -y zstd
fi

sudo mkdir -p "$PADIS_DATA_MOUNT"
if mountpoint -q "$PADIS_DATA_MOUNT"; then
  echo "$PADIS_DATA_MOUNT is already mounted."
else
  sudo chown "$(id -u):$(id -g)" "$PADIS_DATA_MOUNT"
  gcsfuse --implicit-dirs "$PADIS_GCS_BUCKET" "$PADIS_DATA_MOUNT"
fi
mountpoint "$PADIS_DATA_MOUNT"
ls -la "$PADIS_DATA_MOUNT" | sed -n '1,40p'

## Locate or clone LION

The checkout path is `/content/LION`, so repo operations and editable installs happen on Colab's local disk rather than through `gcsfuse`. By default this clones your fork and checks out `feature/PaDIS_Implementation`, then fetches and fast-forwards it to the newest remote version. If the repo or branch is private, add a Colab secret named `GITHUB_TOKEN` or paste a token into the config cell before running this cell.

In [ ]:
%%bash
set -euo pipefail
source /content/padis_colab_env.sh

clone_url="$LION_GIT_URL"
if [ -n "${GITHUB_TOKEN:-}" ] && [[ "$clone_url" == https://github.com/* ]]; then
  clone_url="${clone_url/https:\/\/github.com\//https:\/\/x-access-token:${GITHUB_TOKEN}@github.com/}"
fi

if [ ! -d "$LION_ROOT/.git" ]; then
  mkdir -p "$(dirname "$LION_ROOT")"
  rm -rf "$LION_ROOT"
  git clone --recursive "$clone_url" "$LION_ROOT"
fi

cd "$LION_ROOT"
git remote set-url origin "$clone_url"
git fetch --all --tags --prune
if [ -n "$LION_GIT_REF" ]; then
  if git show-ref --verify --quiet "refs/remotes/origin/$LION_GIT_REF"; then
    git checkout -B "$LION_GIT_REF" "origin/$LION_GIT_REF"
  else
    git checkout "$LION_GIT_REF"
  fi
else
  git pull --ff-only
fi
if [ -n "${GITHUB_TOKEN:-}" ] && [[ "$LION_GIT_URL" == https://github.com/* ]]; then
  git remote set-url origin "$LION_GIT_URL"
fi
git submodule sync --recursive
git submodule update --init --recursive
git status --short
test -x scripts/paper_scripts/PaDIS-Reproduction/platforms/gcp/run_PaDIS_GCP_manual_reconstruction.sh
echo "LION checkout is ready at $LION_ROOT."

## Install conda and the LION environment

This follows the repository setup instructions: create `lion` from `env_base.yml`, install CUDA PyTorch, then install LION editable from the checkout. The conda installation lives under `/content`, so it is fast but does not persist after the Colab runtime is destroyed.

In [ ]:
%%bash
set -euo pipefail
source /content/padis_colab_env.sh

if [ ! -x "$MINIFORGE_ROOT/bin/conda" ]; then
  wget -qO /content/miniforge.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
  bash /content/miniforge.sh -b -p "$MINIFORGE_ROOT"
fi

source "$MINIFORGE_ROOT/etc/profile.d/conda.sh"
conda config --set channel_priority flexible

if conda env list | awk '{print $1}' | grep -qx "$LION_CONDA_ENV"; then
  echo "Updating existing conda env $LION_CONDA_ENV."
  conda env update --name "$LION_CONDA_ENV" --file "$LION_ROOT/env_base.yml" --prune
else
  conda env create --file="$LION_ROOT/env_base.yml" --name="$LION_CONDA_ENV"
fi

conda activate "$LION_CONDA_ENV"
python -m pip install --upgrade 'pip<=25.2'
python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
python -m pip install -e "$LION_ROOT"

python - <<'PY'
import torch
import LION
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
print('LION import ok')
PY

## Authenticate to WandB

If you want fallback training for missing reconstruction checkpoints to log online and upload artifacts, add a Colab secret named `WANDB_API_KEY` before running the config cell. Without it, run `wandb login` manually in a terminal/cell before the reconstruction runner needs to train a missing checkpoint.

In [ ]:
%%bash
set -euo pipefail
source /content/padis_colab_env.sh
source "$MINIFORGE_ROOT/etc/profile.d/conda.sh"
conda activate "$LION_CONDA_ENV"

if [ -n "${WANDB_API_KEY:-}" ]; then
  wandb login --relogin "$WANDB_API_KEY"
else
  echo "WANDB_API_KEY is not set. Existing WandB status follows; login manually if needed."
  wandb status || true
fi

## Preflight the reconstruction matrix

This previews and writes the expected job manifest. The full runner below trains missing trainable checkpoints first, then performs the strict checkpoint check before launching reconstruction jobs.

In [ ]:
%%bash
set -euo pipefail
source /content/padis_colab_env.sh
source "$MINIFORGE_ROOT/etc/profile.d/conda.sh"
conda activate "$LION_CONDA_ENV"
cd "$LION_ROOT"

export PADIS_TRAIN_ROOT="${PADIS_RUN_ROOT}/final_real_runs/${PADIS_GCP_RUN_NAME}"
export PADIS_RECON_ROOT="${PADIS_RUN_ROOT}/final_real_runs/${PADIS_GCP_RUN_NAME}_reconstruction"
export PADIS_RECON_EXPECTED_JOBS_JSON="${PADIS_RECON_ROOT}/reconstruction_matrix_jobs.json"
mkdir -p "$PADIS_RECON_ROOT"

python -u scripts/paper_scripts/PaDIS-Reproduction/reconstruction/PaDIS_run_reconstruction_matrix.py \
  --training-root "$PADIS_TRAIN_ROOT" \
  --output-root "$PADIS_RECON_ROOT" \
  --checkpoint-policy min_intense_val \
  --job-order gcp_spot \
  --hparam-defaults json \
  --hparam-defaults-json "$LION_ROOT/scripts/paper_scripts/PaDIS-Reproduction/config/reconstruction_hparam_defaults.json" \
  --hparam-run-root "$PADIS_RUN_ROOT/hparam_tuning/runs" \
  --models method_default \
  --methods "$PADIS_RECON_METHODS" \
  --experiments "$PADIS_RECON_EXPERIMENTS" \
  --ablations "$PADIS_RECON_ABLATIONS" \
  --implementations method_default \
  --geometries lion \
  --split test \
  --max-samples "$PADIS_RECON_MAX_SAMPLES" \
  --device cuda \
  --list > "$PADIS_RECON_EXPECTED_JOBS_JSON"

python - <<'PY'
import json, os
path = os.environ['PADIS_RECON_EXPECTED_JOBS_JSON']
jobs = json.load(open(path))
print(f'Prepared {len(jobs)} reconstruction jobs at {path}')
print('First job:', jobs[0]['implementation'], jobs[0]['method'], jobs[0]['experiment'])
print('Last job:', jobs[-1]['implementation'], jobs[-1]['method'], jobs[-1]['experiment'])
PY
sync -f "$PADIS_RECON_ROOT" 2>/dev/null || sync

## Run the resumable inference matrix

This calls the manual GCP runner. It does not remount the bucket because the notebook already mounted it. Rerun this cell after a disconnect; completed jobs are skipped.

In [ ]:
%%bash
set -euo pipefail
source /content/padis_colab_env.sh
source "$MINIFORGE_ROOT/etc/profile.d/conda.sh"
conda activate "$LION_CONDA_ENV"
cd "$LION_ROOT"

export PADIS_MOUNT_BUCKET=0
export PADIS_MANUAL_RECON_SKIP_ENV_ACTIVATE=1
export PADIS_DATA_MOUNT="$PADIS_DATA_MOUNT"
export LION_ROOT="$LION_ROOT"
export LION_DATA_PATH="$LION_DATA_PATH"
export LION_EXPERIMENTS_PATH="$LION_EXPERIMENTS_PATH"
export PADIS_RUN_ROOT="$PADIS_RUN_ROOT"
export PADIS_GCP_RUN_NAME="$PADIS_GCP_RUN_NAME"
export PADIS_RECON_METHODS="$PADIS_RECON_METHODS"
export PADIS_RECON_EXPERIMENTS="$PADIS_RECON_EXPERIMENTS"
export PADIS_RECON_ABLATIONS="$PADIS_RECON_ABLATIONS"
export PADIS_RECON_MAX_SAMPLES="$PADIS_RECON_MAX_SAMPLES"
export PADIS_RECON_TASKS_PER_GPU="$PADIS_RECON_TASKS_PER_GPU"
export PADIS_RECON_SAVE_PREVIEWS="$PADIS_RECON_SAVE_PREVIEWS"
export PADIS_RECON_PROG_BAR="$PADIS_RECON_PROG_BAR"
export PADIS_WANDB_PROJECT="$PADIS_WANDB_PROJECT"
export PADIS_WANDB_MODE="$PADIS_WANDB_MODE"
export PADIS_NO_WANDB="$PADIS_NO_WANDB"
export PADIS_NO_WANDB_ARTIFACT="$PADIS_NO_WANDB_ARTIFACT"
export WANDB_API_KEY="${WANDB_API_KEY:-}"

bash scripts/paper_scripts/PaDIS-Reproduction/platforms/gcp/run_PaDIS_GCP_manual_reconstruction.sh

## Inspect progress

Run this cell any time to see completed/failed jobs and recent logs.

In [ ]:
%%bash
set -euo pipefail
source /content/padis_colab_env.sh
RECON_ROOT="${PADIS_RUN_ROOT}/final_real_runs/${PADIS_GCP_RUN_NAME}_reconstruction"
STATE_DIR="$RECON_ROOT/.manual_gcp_reconstruction"
echo "State: $STATE_DIR"
find "$STATE_DIR/done" -maxdepth 1 -type f -name '*.done' 2>/dev/null | wc -l | xargs echo done:
find "$STATE_DIR/failed" -maxdepth 1 -type f -name '*.failed' 2>/dev/null | wc -l | xargs echo failed:
find "$STATE_DIR/running" -maxdepth 1 -type f -name '*.running' 2>/dev/null | wc -l | xargs echo running:
echo
echo "Recent failed markers:"
find "$STATE_DIR/failed" -maxdepth 1 -type f -name '*.failed' -printf '%TY-%Tm-%Td %TH:%TM %p\n' 2>/dev/null | sort | tail -n 10 || true
echo
echo "Recent logs:"
find "$STATE_DIR/logs" -maxdepth 1 -type f -name '*.log' -printf '%TY-%Tm-%Td %TH:%TM %p\n' 2>/dev/null | sort | tail -n 10 || true